In [60]:
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import os
import json
import ROOT

In [61]:
basedir = os.path.join(os.environ.get('ttH_yy_DIR'), 'final')

process = {
    'ttHyy' : { 'sample_list' : ['mgp8_pp_tth01j_5f_haa'], 
               'label' : r'$ttH \rightarrow \gamma\gamma$'},
    'yy_jets' : { 'sample_list' : ['mgp8_pp_jjaa_5f'], 
               'label' : r'$\gamma\gamma$ + jets'},
    'ttyy' : { 'sample_list' : ['mgp8_pp_ttaa_semilep_5f_100TeV'], 
               'label' : '$tt\gamma\gamma$'}
}

selection = {
            "nocuts" : "All events", # all events
            "photons": "$\geq$ 2 photons",
            "photons_rel_pt": "Rel. $p_{T}$ cuts",
            "photons_myy_window": "105 < $m_{\gamma\gamma}$ < 160 GeV",
            "preselection": "$\geq$ 2 b-jets",
            "lep_channel" : "$\geq$ 1 lepton",
            #"preselection_myy_window_narrow": "120 < $m_{\gamma\gamma}$ < 130 GeV",
}

variables = ['weight']

process_infos = "/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v06_II.json"

lumi = 3e7 # 3 * 10^7 pb-1 = 3 ab-1

In [62]:
with open(process_infos, 'r') as f :
    process_dict = json.load(f)

In [63]:
df = {}
sow = {}

In [64]:
for p in process.keys() :
    df[p] = {}
    print(p)
    for s in selection.keys() :
        df1 = []
        sow[p] = []
        print("\t"+s)
        for sample in process[p]['sample_list'] :
            print("\t\t"+sample)
            inputfile = os.path.join(basedir, sample+"_"+s+".root")
            # get sum of weights
            f = ROOT.TFile.Open(inputfile)
            sow[p].append(f.Get("SumOfWeights").GetVal())
            f.Close()
            # get sample
            with uproot.open(inputfile) as f :
                df1.append(ak.to_dataframe(f['events'].arrays(expressions=variables, library='ak')))
                df1[-1]["weight"] = df1[-1]["weight"]/sow[p][-1]*process_dict[sample]['crossSection']*process_dict[sample]['kfactor']*process_dict[sample]['matchingEfficiency']
        df[p][s] = pd.concat(df1, copy=True, ignore_index=True)

ttHyy
	nocuts
		mgp8_pp_tth01j_5f_haa
	photons
		mgp8_pp_tth01j_5f_haa
	photons_rel_pt
		mgp8_pp_tth01j_5f_haa
	photons_myy_window
		mgp8_pp_tth01j_5f_haa
	preselection
		mgp8_pp_tth01j_5f_haa
	lep_channel
		mgp8_pp_tth01j_5f_haa
yy_jets
	nocuts
		mgp8_pp_jjaa_5f
	photons
		mgp8_pp_jjaa_5f
	photons_rel_pt
		mgp8_pp_jjaa_5f
	photons_myy_window
		mgp8_pp_jjaa_5f
	preselection
		mgp8_pp_jjaa_5f
	lep_channel
		mgp8_pp_jjaa_5f
ttyy
	nocuts
		mgp8_pp_ttaa_semilep_5f_100TeV
	photons
		mgp8_pp_ttaa_semilep_5f_100TeV
	photons_rel_pt
		mgp8_pp_ttaa_semilep_5f_100TeV
	photons_myy_window
		mgp8_pp_ttaa_semilep_5f_100TeV
	preselection
		mgp8_pp_ttaa_semilep_5f_100TeV
	lep_channel
		mgp8_pp_ttaa_semilep_5f_100TeV


In [65]:
my_entries = {
    "Selection" : []
}

In [66]:
for p in process.keys() :
    my_entries[process[p]['label']] = []

In [67]:
for s in selection :
    my_entries["Selection"].append(selection[s])
    for p in process.keys() :
        my_entries[process[p]['label']].append(len(df[p][s]))

In [68]:
my_entries = pd.DataFrame(my_entries)
my_entries = my_entries.set_index('Selection')

In [69]:
my_entries

,$ttH \rightarrow \gamma\gamma$,$\gamma\gamma$ + jets,$tt\gamma\gamma$
Selection,,,
All events,306353,2470000,50000
$\geq$ 2 photons,175194,1394811,21815
Rel. $p_{T}$ cuts,145445,1284467,14579
105 < $m_{\gamma\gamma}$ < 160 GeV,99050,1197377,6579
$\geq$ 2 b-jets,61545,34527,3568
$\geq$ 1 lepton,16901,49,1879


In [70]:
my_yields = {
    "Selection" : []
}

In [71]:
for p in process.keys() :
    my_yields[process[p]['label']] = []

In [72]:
for s in selection :
    my_yields["Selection"].append(selection[s])
    for p in process.keys() :
        my_yields[process[p]['label']].append(df[p][s]["weight"].sum()*lumi)

In [73]:
my_yields = pd.DataFrame(my_yields)
my_yields = my_yields.set_index('Selection')

In [74]:
my_yields

,$ttH \rightarrow \gamma\gamma$,$\gamma\gamma$ + jets,$tt\gamma\gamma$
Selection,,,
All events,2.283668e+06,6.469200e+08,8.556000e+06
$\geq$ 2 photons,1.306010e+06,3.653162e+08,3.733119e+06
Rel. $p_{T}$ cuts,1.084243e+06,3.364159e+08,2.494859e+06
105 < $m_{\gamma\gamma}$ < 160 GeV,7.384012e+05,3.136061e+08,1.125825e+06
$\geq$ 2 b-jets,4.588242e+05,9.042999e+06,6.105752e+05
$\geq$ 1 lepton,1.259919e+05,1.283364e+04,3.215444e+05


In [75]:
my_eff = {
    "Selection" : []
}

In [76]:
for p in process.keys() :
    my_eff[process[p]['label']] = []

In [77]:
for s in selection :
    my_eff["Selection"].append(selection[s])
    for p in process.keys() :
        my_eff[process[p]['label']].append(df[p][s]["weight"].sum()/df[p]["nocuts"]["weight"].sum())

In [78]:
my_eff = pd.DataFrame(my_eff)
my_eff = my_eff.set_index('Selection')

In [79]:
my_eff

,$ttH \rightarrow \gamma\gamma$,$\gamma\gamma$ + jets,$tt\gamma\gamma$
Selection,,,
All events,1.000000,1.000000,1.000000
$\geq$ 2 photons,0.571891,0.564701,0.436316
Rel. $p_{T}$ cuts,0.474782,0.520027,0.291592
105 < $m_{\gamma\gamma}$ < 160 GeV,0.323340,0.484768,0.131583
$\geq$ 2 b-jets,0.200915,0.013979,0.071362
$\geq$ 1 lepton,0.055171,0.000020,0.037581
